# Evaluation warning

This notebook is retained as exploratory history. Its original model scores are likely optimistic because the workflow used random row splits and/or SMOTE before splitting. Use `../../scripts/evaluate_models.py` and see `../../docs/modeling/leakage_audit.md` for leakage-aware evaluation.


In [ ]:
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import model_from_json
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import backend as K
import pandas as pd
import numpy as np
import warnings
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from keras.wrappers.scikit_learn import KerasClassifier

In [ ]:
df_adb = pd.read_csv('time_series_adb.csv')
df_non_adb = pd.read_csv('time_series_non_adb.csv')

df_adb_flipped = pd.DataFrame()
warnings.filterwarnings("ignore")

# Iterate over every 300 columns
for i in range(0, df_adb.shape[1], 300):
    # Get the range of columns to flip
    columns_to_flip = df_adb.columns[i:i+300]
    
    # Flip the order of columns and store in the new DataFrame
    df_adb_flipped[columns_to_flip[::-1]] = df_adb[columns_to_flip]
df_adb_flipped['adb'] = 1
df_adb_final = df_adb_flipped.iloc[:, 600:]

df_non_adb_flipped = pd.DataFrame()
for i in range(0, df_non_adb.shape[1], 300):
    # Get the range of columns to flip
    columns_to_flip = df_non_adb.columns[i:i+300]
    
    # Flip the order of columns and store in the new DataFrame
    df_non_adb_flipped[columns_to_flip[::-1]] = df_non_adb[columns_to_flip]
    
df_non_adb_flipped['adb'] = 0
df_non_adb_final = df_non_adb_flipped.iloc[:, 600:]
df = pd.concat([df_adb_final,df_non_adb_final], axis=0)
df = df.dropna()
print(df)


In [ ]:
x = df.iloc[:, 0:7800].values
y = df['adb'].values

train_ratio = 0.8
test_ratio = 0.2

smote = SMOTE(random_state=123)
x_train1, y_train1 = smote.fit_resample(x, y)

x_train, x_test, y_train, y_test = train_test_split(x_train1, y_train1, test_size=test_ratio, train_size=train_ratio, random_state=129)



print('train')
print('dataset:', len(x_train))
print('have:', sum(y_train))
print('No:', len(y_train) - sum(y_train))
print('------------')
print('test')
print('dataset:', len(x_test))
print('have:', sum(y_test))
print('No:', len(y_test) - sum(y_test))

x_train_array = np.array(x_train)
# Reshape x_train_array to (num_samples, num_timesteps, num_features)
num_samples_train = x_train_array.shape[0]
num_timesteps = 300
num_features = 26

X_train = x_train_array.reshape(num_samples_train, num_timesteps, num_features)
Y_train = y_train

x_test_array = np.array(x_test)
num_samples_test = x_test_array.shape[0]

X_test = x_test_array.reshape(num_samples_test, num_timesteps, num_features)
Y_test = y_test

print(X_train.shape)
print(X_test.shape)

def GRU_model(shape):
    K.clear_session()
    model = Sequential()

    # Add a LSTM layer with 128 internal units.
    model.add(layers.InputLayer(input_shape=(shape[1], shape[2])))
    model.add(layers.GRU(units=256, return_sequences=True, dropout=0.2, activation='tanh')) # change units dropout less maybe? 
    model.add(layers.Dropout(0.8))  # try 0.8, 0.9
    model.add(layers.GRU(units=256, activation='tanh'))
    model.add(layers.Dense(1, activation='tanh'))
    model.compile(loss='binary_crossentropy', optimizer='RMSprop', metrics=['accuracy'])
    return model


def load_trained_model(X_train, weights_path):
    model = create_model(X_train.shape)
    model.load_weights(weights_path)
    return model

GRU = GRU_model(X_train.shape)
GRU.summary()

In [ ]:
# RNN with K-fold
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=113)
loss, Acc, val_loss, val_accuracy = [], [], [], []
es = EarlyStopping(monitor='val_loss', patience=3)
GRU = GRU_model(X_train.shape)
cvscores = []

for train, test in kfold.split(X_train, Y_train):
    X_train[train]

    his = GRU.fit(X_train[train], Y_train[train], verbose=0,
                  validation_data=(X_train[test], Y_train[test]),
                  callbacks=[es])


    loss.append(his.history['loss'])
    Acc.append(his.history['accuracy'])
    val_loss.append(his.history['val_loss'])
    val_accuracy.append(his.history['val_accuracy'])

    # evaluate the model
    scores = GRU.evaluate(X_train[test], Y_train[test], verbose=0)

    print("%s: %.2f%%" % (GRU.metrics_names[1], scores[1]*100))
    cvscores.append(scores[1])

print("%.2f%% (+/- %.2f%%)" % (np.mean(cvscores)*100, np.std(cvscores)*100))

In [ ]:
# Test the model using the test data
test_loss, test_accuracy = GRU.evaluate(X_test, Y_test, verbose=0)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Predict on the test data
y_pred = GRU.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)  # Convert probabilities to binary predictions

# Calculate evaluation metrics
accuracy = accuracy_score(Y_test, y_pred_binary)
precision = precision_score(Y_test, y_pred_binary)
roc_auc = roc_auc_score(Y_test, y_pred)
f1 = f1_score(Y_test, y_pred_binary)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("ROC AUC:", roc_auc)
print("F1 Score:", f1)

In [ ]:
cm = confusion_matrix(y_test, y_pred_binary)

print("Confusion Matrix:")
print(cm)